# Bài tập về nhà  
Giao diện cho bài toán máy hút bụi với các thuật toán tìm kiếm:  

**Buổi 9:**

- Leo đồi  
    Cost tính bằng h(n), cài đặt bằng khoảng cách Manhattan từ robot đến bụi XA NHẤT
    + [Leo đồi khởi tạo ngẫu nhiên](#thuật-toán-leo-đồi-khởi-tạo-ngẫu-nhiên)  
        Cài đặt sẵn trong thuật toán MAX_RESTART = 10  

Ở buổi 5 đã xây dựng các thuật toán:
- BFS cách tiếp cận 1  
- BFS cách tiếp cận 2  
- DFS cách tiếp cận 1  
- DFS cách tiếp cận 2  

Ở buổi 6 đã xây dựng các thuật toán:  
- IDS cách tiếp cận 1  
- IDS cách tiếp cận 2  
- UCS  

Ở buổi 7 đã xây dựng các thuật toán:  
- Greedy Search  
- A*  
  
Ở buổi 8 đã cây dựng các thuật toán:  
- IDA*  
- Leo đồi:  
    + Leo đồi đơn giản  
    + Leo đồi dốc nhất  
    + Leo đồi ngẫu nhiên   

**Link github:**  

**BT cá nhân:** Mỗi thuật toán tổ chức thành các module, file gui.ipynb sẽ import các thuật toán và gọi nó khi giao diện chạy thuật toán.  
https://github.com/Duc-Luong060106/vacuum_cleaner_personal_project

In [16]:
# Class định nghĩa các thông tin của trạng thái
class Node:
    def __init__(self, state, parent, action, path_cost):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

In [ ]:
# Trả về list là chuỗi đường đi đến node truyền vào
def get_solution(node):
    path = []

    while node != None:
        path.append(node)
        node = node.parent

    path.reverse()
    return path


def print_solution(result):
    if result == None:
        print("Không tìm thấy đường đi!!!")
    else:
        for idx, node in enumerate(result):
            print(f"Bước {idx}: {node.action if node.action != None else "Khởi tạo"}")
            for row in node.state:
                print(*row)
            
            print()

        print("Phòng đã được dọn dẹp!!!")

In [18]:
# So sánh trạng thái với trạng thái đích (Không còn bụi)
def compare_state(state, goal_state):
    for x in range(len(state)):
        for y in range(len(state[0])):
            # Vị trí của robot đang đứng đã hút sạch bụi
            if state[x][y] != 'X' and state[x][y] != goal_state[x][y]:
                return False

    return True

In [19]:
def find_vacuum_position(state):
    for x in range(len(state)):
        for y in range(len(state[0])):

            if state[x][y] == 'X':
                return x, y
    
    return None

# Tạo ra các trạng thái con hợp lệ và hành động để sinh ra các trạng thái đó
def gen_actions(state):
    m = len(state)
    n = len(state[0])

    x, y = find_vacuum_position(state)

    actions = []

    if x > 0 and state[x-1][y] != 2:
        up_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x-1}][{y}]" if up_state[x-1][y] == 1 else ""

        up_state[x][y], up_state[x-1][y] = 0, 'X'

        actions.append(("Up" + clean_action, up_state))

    if x < m - 1 and state[x+1][y] != 2:
        down_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x+1}][{y}]" if down_state[x+1][y] == 1 else ""

        down_state[x][y], down_state[x+1][y] = 0, 'X'

        actions.append(("Down" + clean_action, down_state))

    if y > 0 and state[x][y-1] != 2:
        left_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y-1}]" if left_state[x][y-1] == 1 else ""

        left_state[x][y], left_state[x][y-1] = 0, 'X'

        actions.append(("Left" + clean_action, left_state))

    if y < n - 1 and state[x][y+1] != 2:
        right_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y+1}]" if right_state[x][y+1] == 1 else ""

        right_state[x][y], right_state[x][y+1] = 0, 'X'

        actions.append(("Right" + clean_action, right_state))

    return actions

## Thuật toán Leo đồi khởi tạo ngẫu nhiên  

In [20]:
# Nhóm thuật toán này sử dụng chi phí heuristic (h(n)), được cài đặt là khoảng cách heuristic XA NHẤT từ robot đến rác.
def calculate_heuristic_cost(state):
    x, y = find_vacuum_position(state)
    heuristic_cost = 0

    for i in range(len(state)):
        for j in range(len(state[0])):
            if state[i][j] == 1:
                manhattan_distance = abs(x-i) + abs(y-j)
                heuristic_cost = max(heuristic_cost, manhattan_distance) if heuristic_cost != 0 else manhattan_distance 
    
    return heuristic_cost

In [ ]:
import random

def randomm_restart_hill_climbing(initial, goal):
    MAX_RESTART = 10    # Cài đặt sẵn MAX_RESTART = 10

    for i in range(1, MAX_RESTART + 1):
        current_node = Node(initial, None, None, calculate_heuristic_cost(initial))
    
        while True:
            if compare_state(current_node.state, goal):
                return get_solution(current_node)

            better_neighbor = []

            for action, child_state in gen_actions(current_node.state):
                child_cost = calculate_heuristic_cost(child_state)

                if child_cost < current_node.path_cost:
                    better_neighbor.append((action, child_state, child_cost))
                    
            if better_neighbor:
                action, child_state, child_cost = random.choice(better_neighbor)
                current_node = Node(child_state, current_node, action, child_cost)
                
            else:
                break
    
    return None

In [25]:
# Test thử 
start =  [['X', 1, 1],
          [0, 2, 1],
          [2, 2, 1]]

goal = [[0, 0, 0],
        [0, 2, 0],
        [2, 2, 0]] 

result = randomm_restart_hill_climbing(start, goal)

print_solution(result)

Bước 0: Khởi tạo
X 1 1
0 2 1
2 2 1

Bước 1: Right và dọn rác ô [0][1]
0 X 1
0 2 1
2 2 1

Bước 2: Right và dọn rác ô [0][2]
0 0 X
0 2 1
2 2 1

Bước 3: Down và dọn rác ô [1][2]
0 0 0
0 2 X
2 2 1

Bước 4: Down và dọn rác ô [2][2]
0 0 0
0 2 0
2 2 X

Phòng đã được dọn dẹp!!!
